query, product_title, relevance_label

https://huggingface.co/docs/peft/v0.8.0/en/task_guides/semantic-similarity-lora

https://medium.com/@venkat.ramrao/fine-tuning-a-sentence-transformer-for-semantic-search-7c7a57f4db2f

https://medium.com/@kelvin.lu.au/fine-tuning-embedding-model-with-peft-and-lora-3b6f08987c24

# Import data

In [5]:
from pymongo import MongoClient
import pandas as pd

# MongoDB connection details
mongo_uri = "mongodb://localhost:27017/"
db_name = "task1"
collection_name = "bbc_news"

# Connect to MongoDB
client = MongoClient(mongo_uri)
db = client[db_name]
collection = db[collection_name]

documents = list(collection.find())
df = pd.DataFrame(documents)
client.close()

df.head()

,_id,title,pubDate,guid,link,description,content,openai_embedding
0,671e2d0aeccd4a5f4883e981,Justin Welby: Political leaders should treat o...,2024-01-01 00:00:04,https://www.bbc.co.uk/news/uk-67844356,https://www.bbc.co.uk/news/uk-67844356?at_medi...,The Archbishop of Canterbury urges politicians...,Justin Welby: Political leaders should treat o...,"[0.02301846444606781, -0.006470758933573961, 0..."
1,671e2d0aeccd4a5f4883e982,Almost three million tested for cancer in England,2024-01-01 00:09:56,https://www.bbc.co.uk/news/health-67841348,https://www.bbc.co.uk/news/health-67841348?at_...,Record numbers are being tested for cancer but...,Almost three million tested for cancer in Engl...,"[0.0134049067273736, 0.012411133386194706, 0.0..."
2,671e2d0aeccd4a5f4883e983,Household energy price rise of 5% comes into f...,2024-01-01 00:00:16,https://www.bbc.co.uk/news/business-67785266,https://www.bbc.co.uk/news/business-67785266?a...,A higher cap for the next three months adds £9...,Household energy price rise of 5% comes into f...,"[-0.00222824071533978, 0.032821644097566605, 0..."
3,671e2d0aeccd4a5f4883e984,Primrose Hill stabbing: Harry Pitman named as ...,2024-01-01 17:11:13,https://www.bbc.co.uk/news/uk-england-london-6...,https://www.bbc.co.uk/news/uk-england-london-6...,"Harry Pitman, 16, was attacked on London's Pri...",Primrose Hill stabbing: Harry Pitman named as ...,"[-0.0027212619315832853, 0.003952309023588896,..."
4,671e2d0aeccd4a5f4883e985,Israel Supreme Court strikes down judicial ref...,2024-01-01 19:47:58,https://www.bbc.co.uk/news/world-middle-east-6...,https://www.bbc.co.uk/news/world-middle-east-6...,The controversial plans triggered nationwide p...,Israel Supreme Court strikes down judicial ref...,"[0.026911897584795952, 0.025074539706110954, 0..."


Save the content to csv to let ChatGPT create the queries

In [4]:
df['content'].to_csv('notebooks_data/news_content.csv', index=False)

## Query data

Queries created by ChatGPT

In [11]:
gpt_queries = pd.read_csv('notebooks_data/user_queries.csv')
gpt_queries

,user_query
0,What is the latest update on politics?
1,Can you explain the recent developments in pol...
2,How has politics changed in recent news?
3,What is the current situation with politics?
4,What are the key takeaways from the recent new...
...,...
4995,What are the benefits of the recent changes in...
4996,Could you summarize the controversy around edu...
4997,Could you provide an overview of the recent ev...
4998,What are the key takeaways from the recent new...


In [12]:
gpt_queries['user_query'].duplicated().sum()

np.int64(4692)

In [15]:
unique_gpt_queries = gpt_queries.drop_duplicates(subset='user_query')
print(len(unique_gpt_queries))
print(unique_gpt_queries['user_query'].duplicated().sum())

308
0


In [16]:
unique_gpt_queries.to_csv('notebooks_data/user_queries_unique.csv', index=False)

### Using ChatGPT to create 2K content query examples

Create 2K random news content dataset

In [25]:
random_news = df.sample(n=5).reset_index(drop=True)
for index, content in random_news['content'].items():
    print(f'"{content}"')

"Scotland fans endure humbling Euro  opener against Germany. The hosts defeated Steve Clarke's men 5-1 in the opening match of the tournament in Munich."
"Explosions hit Odesa as Zelensky meets Greek PM. Ukraine's navy says five people have been killed after missile strikes in the southern port city."
"Smith beats Littler and Price on way to opening Premier League win. Michael Smith beats Gerwyn Price to win the opening night of the 2024 Premier League Darts in Cardiff."
"Israel officials support Gaza destruction, court hears. South African lawyers present their case on the first day of a hearing at the UN's top court."
"Police investigate 'care of dead' at funeral homes. An investigation begins after "a report received of concern for care of the deceased"."


In [22]:
"""
RSS news content:
<news>"Scotland fans endure humbling Euro opener against Germany. The hosts defeated Steve Clarke's men 5-1 in the opening match of the tournament in Munich."</news>
<query1>"How did Scotland perform in their Euro opener against Germany?"</query1>
<query2>"What challenges is Scotland facing in the current Euro tournament?"</query2>
<news>"Explosions hit Odesa as Zelensky meets Greek PM. Ukraine's navy says five people have been killed after missile strikes in the southern port city."</news>
<query1>"What were the casualties from the recent missile strikes in Odesa?"</query1>
<query2>"How is Ukraine handling the conflict in its southern port cities?"</query2>
<news>"Israel officials support Gaza destruction, court hears. South African lawyers present their case on the first day of a hearing at the UN's top court."</news>
<query1>"What did Israel officials testify regarding Gaza at the UN court hearing?"</query1>
<query2>"What legal actions are being taken internationally concerning Gaza?"</query2>
<news>"Smith beats Littler and Price on way to opening Premier League win. Michael Smith beats Gerwyn Price to win the opening night of the 2024 Premier League Darts in Cardiff."</news>
<query1>"Who won the opening night of the 2024 Premier League Darts?"</query1>
<query2>"How has Michael Smith's performance been in recent darts tournaments?"</query2>
<news>"'Positives for Liverpool - but a big chance missed'. MOTD2 pundit Danny Murphy says Liverpool's draw at Old Trafford is not the end of the world for their title hopes - but they should have secured all three points."</news>
<query1>"What were Danny Murphy's comments on Liverpool's draw at Old Trafford?"</query1>
<query2>"What are the implications of Liverpool's recent performances for their title hopes?"</query2>
"""


'\nRSS news content:\n<news>"Steve Rosenberg on Russia\'s stage-managed election. In Borovsk, Steve Rosenberg looks at the Russia Putin wants you to see - and Russia in reality."</news>\n"Serving Met Police officer in court charged with rape. The Met says the case relates to a report a man was raped at a home in north London last Monday."\n"Who will replace McConnell as top Senate Republican?. The three "Johns" and a senator who challenged Mr McConnell for his post are considered likely successors."\n"King\'s official Coronation scroll is first without animal skin. For 700 years there has been a handwritten account of coronations, but there are some names missing."\n"\'Positives for Liverpool - but a big chance missed\'. MOTD2 pundit Danny Murphy says Liverpool\'s draw at Old Trafford is not the end of the world for their title hopes - but they should have secured all three points."\n'

In [37]:
random_news = df.sample(n=5).reset_index(drop=True)
for index, content in random_news['content'].items():
    print(f'"{content}"')

"North Sea oil and gas claims fact-checked. The government wants to guarantee annual oil and gas licensing rounds to "improve energy security"."
"Pupils injured in crush at school gate. The head teacher says a "surge" developed when the gate was not opened at the end of the school day."
"Sixth person charged with spying for Russia in UK. Bulgarian national Tihomir Ivanov Ivanchev, 38, will appear in court on Wednesday."
"'We went viral as Oompa Loompas but we're just normal people'. Kirsty Paterson and Jenny Fogarty were hired to work at the Willy Wonka Chocolate Experience in Glasgow."
"BBC Proms conductor Sir Andrew Davis dies aged 80. Sir Andrew was one of the longest-serving chief conductors of the BBC Symphony Orchestra."


In [31]:
from openai import OpenAI

client = OpenAI(api_key='<API_KEY>')
def ask_chatgpt_4o_mini(prompt: str) -> str:
    """
    Use the OpenAI ChatGPT-4o-mini to generate an answer based on the given prompt.

    Args:
        prompt (str): The prompt for the chatbot.

    Returns:
        str: The generated answer.
    """
    completion = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "user", "content": prompt}
        ]
    )

    return completion.choices[0].message.content.strip()

In [34]:
import re

def extract_queries(model_response):
    # Regular expressions to extract content within <query1> and <query2> tags
    query1_match = re.search(r'<query1>"(.*?)"</query1>', model_response)
    query2_match = re.search(r'<query2>"(.*?)"</query2>', model_response)
    
    # Extract the matched content if available
    query1 = query1_match.group(1) if query1_match else None
    query2 = query2_match.group(1) if query2_match else None
    
    return query1, query2

In [38]:
random_news = df.sample(n=1000).reset_index(drop=True)

In [40]:
import pandas as pd

# Initialize an empty list to gather results
results = []

# Loop through each row in the `random_news` DataFrame
for index, row in random_news.iterrows():
    news_content = row['content']
    
    # Create the prompt
    prompt = f"""
I have a dataset of news content, and I need to generate two user queries for each news article. The queries should be relevant to the news content and answerable based on it. 
Here is the format I need for each news article:

<news>"[NEWS_CONTENT]"</news>
<query1>"[QUERY_1]"</query1>
<query2>"[QUERY_2]"</query2>

Guidelines for creating queries:
1. Make the first query closely related to the specific details of the news.
2. Make the second query relevant to the main topic but broader, covering general aspects of the subject.

Please provide two queries in the exact format shown for each news article. Below are some examples to follow:

<news>"Scotland fans endure humbling Euro opener against Germany. The hosts defeated Steve Clarke's men 5-1 in the opening match of the tournament in Munich."</news>
<query1>"How did Scotland perform in their Euro opener against Germany?"</query1>
<query2>"How are going the Euro games?"</query2>
<news>"Explosions hit Odesa as Zelensky meets Greek PM. Ukraine's navy says five people have been killed after missile strikes in the southern port city."</news>
<query1>"What were the casualties from the recent missile strikes in Odesa?"</query1>
<query2>"Ukraine casualties in the war?"</query2>

Now, for this news content:

<news>"{news_content}"</news>

Generate:
<query1>"</query1>
<query2>"</query2>
"""
    
    # Call the function to get the model's response
    model_response = ask_chatgpt_4o_mini(prompt)
    
    # Extract queries from the model's response
    query1, query2 = extract_queries(model_response)
    
    # Append each query as a separate row in results
    if query1:
        results.append({'query': query1, 'news_content': news_content})
    if query2:
        results.append({'query': query2, 'news_content': news_content})
    
    # Print the progress
    if (index + 1) % 10 == 0:  # Adjust the modulus value to print more or less frequently
        percent_complete = ((index + 1) / len(random_news)) * 100
        print(f"Processed {percent_complete:.0f}% -> {index + 1} out of {len(random_news)} rows.")

# Convert the list of results to a DataFrame
queries_df = pd.DataFrame(results)

# Save the DataFrame to a CSV file
queries_df.to_csv('notebooks_data/generated_queries.csv', index=False)

print("Query generation and saving to CSV completed.")


Processed 1% -> 10 out of 1000 rows.
Processed 2% -> 20 out of 1000 rows.
Processed 3% -> 30 out of 1000 rows.
Processed 4% -> 40 out of 1000 rows.
Processed 5% -> 50 out of 1000 rows.
Processed 6% -> 60 out of 1000 rows.
Processed 7% -> 70 out of 1000 rows.
Processed 8% -> 80 out of 1000 rows.
Processed 9% -> 90 out of 1000 rows.
Processed 10% -> 100 out of 1000 rows.
Processed 11% -> 110 out of 1000 rows.
Processed 12% -> 120 out of 1000 rows.
Processed 13% -> 130 out of 1000 rows.
Processed 14% -> 140 out of 1000 rows.
Processed 15% -> 150 out of 1000 rows.
Processed 16% -> 160 out of 1000 rows.
Processed 17% -> 170 out of 1000 rows.
Processed 18% -> 180 out of 1000 rows.
Processed 19% -> 190 out of 1000 rows.
Processed 20% -> 200 out of 1000 rows.
Processed 21% -> 210 out of 1000 rows.
Processed 22% -> 220 out of 1000 rows.
Processed 23% -> 230 out of 1000 rows.
Processed 24% -> 240 out of 1000 rows.
Processed 25% -> 250 out of 1000 rows.
Processed 26% -> 260 out of 1000 rows.
Proc

# Fine tuning E5

In [6]:
import pandas as pd
from sklearn.model_selection import train_test_split

queries_df = pd.read_csv('notebooks_data/generated_queries.csv')
train_df, test_df = train_test_split(queries_df, test_size=0.3, random_state=42)
print(f"Training set size: {len(train_df)}")
print(f"Test set size: {len(test_df)}")

Training set size: 1399
Test set size: 600


## Load the Model and Tokenizer with 8-Bit Quantization

In [7]:
from transformers import AutoTokenizer, AutoModel
from peft import LoraConfig, get_peft_model

# Load tokenizer and model without quantization
model_name = "intfloat/e5-large-v2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)  # Standard loading, no quantization

## Configure QLoRA

In [8]:
# Define QLoRA configuration
lora_config = LoraConfig(
    r=8,               # Rank of the low-rank matrices
    lora_alpha=8,      # Scaling factor for LoRA updates
    target_modules=["key", "query", "value"],  # Adjust based on model structure if necessary
    lora_dropout=0.1,   # Dropout for LoRA layers
    bias="none"         # Exclude biases to focus on core layers
)

# Wrap the model with QLoRA
model = get_peft_model(model, lora_config)


## Pre-Fine-Tuning Performance Evaluation

In [9]:
import torch

def get_embeddings(text, model, tokenizer):
    inputs = tokenizer(text, return_tensors="pt", padding=True, truncation=True).to("cpu")
    with torch.no_grad():
        embeddings = model(**inputs).last_hidden_state.mean(dim=1).cpu().numpy()
    return embeddings


df['e5_embedding_before_tuning'] = df['content'].apply(lambda x: get_embeddings(x, model, tokenizer))

In [10]:
from sklearn.metrics.pairwise import cosine_similarity

def find_top_news(query, df, model, tokenizer, top_k=10):
    query_embedding = get_embeddings(query, model, tokenizer)

    # Calculate cosine similarity between query and all news embeddings
    similarities = df['e5_embedding_before_tuning'].apply(lambda x: cosine_similarity(query_embedding, x)[0][0])
    
    # Add similarity scores to the DataFrame
    df['e5_similarity'] = similarities
    
    # Sort by similarity and return the top_k results
    top_matches = df.nlargest(top_k, 'e5_similarity')
    
    # Clean up by dropping the 'similarity' column
    df.drop(columns=['e5_similarity'], inplace=True)
    
    return top_matches[['content', 'e5_similarity']]

test_queries = ["Products for Color-blind people","JEEP off-road vehicles","What is the origin of Bamba snack","Recommended diet and exercise habits for managing obesity","Keep your children safe on the internet","What are the lates news about AI ethics?","How the war between Israel and Hamas is affecting global politics?","Latest trends in the fashion industry","Interest rates","biden vs trump","Best Marvel movie this year","Northern Lights"]
for user_query in test_queries:
    print(f"\n\nQuery: {user_query}")
    top_news = find_top_news(user_query, df, model, tokenizer)
    for _, row in top_news.iterrows():
                print(f"\033[96m Similarity: {row['e5_similarity']:.3f}::\033[0m {row['content']}")



Query: Products for Color-blind people
 Similarity: 0.785:: Cost of living: How can I make extra income?. Some people have been earning thousands selling clothes and other items since they were in school.
 Similarity: 0.780:: Ethnic bias in health devices 'puts patients at risk'. The review focuses on devices frequently used in the NHS, like pulse oximeters and skin cancer apps.
 Similarity: 0.774:: Americanswers! What is Project 2025? Will Biden take a cognitive test?. We answer your questions on the biggest issues in the run-up to the election
 Similarity: 0.774:: Electioncast: Laura Kuenssberg and Adam Fleming answer your questions. Adam and Laura answer your election questions
 Similarity: 0.773:: The List. Exploring 9 amazing space missions.
 Similarity: 0.773:: Who can I vote for in the general election?. Find out which constituency you are in, who you can vote for and where you can vote using our postcode search.
 Similarity: 0.773:: 'I’ve invented an alternative to uncomforta

In [11]:
import torch
from sklearn.metrics.pairwise import cosine_similarity

# Helper function to get embeddings
def get_embeddings(text, model, tokenizer):
    inputs = tokenizer(text, return_tensors="pt", padding=True, truncation=True).to("cpu")
    with torch.no_grad():
        embeddings = model(**inputs).last_hidden_state.mean(dim=1).cpu().numpy()
    return embeddings

# Calculate pre-fine-tuning similarities
similarities_before = []
for index, row in test_df.iterrows():
    query_embedding = get_embeddings(row['query'], model, tokenizer)
    content_embedding = get_embeddings(row['news_content'], model, tokenizer)
    similarity = cosine_similarity(query_embedding, content_embedding)[0][0]
    similarities_before.append(similarity)

    # Print the progress
    if (index + 1) % 100 == 0:  # Adjust the modulus value to print more or less frequently
        percent_complete = ((index + 1) / len(test_df)) * 100
        print(f"Processed {percent_complete:.0f}% -> {index + 1} out of {len(test_df)} rows.")

# Store pre-fine-tuning similarities
test_df['similarity_before'] = similarities_before
test_df.to_csv('notebooks_data/generated_queries_with_similarity_before.csv', index=False)
print("Average Pre-Fine-Tuning Similarity:", sum(similarities_before) / len(similarities_before))

Processed 300% -> 1800 out of 600 rows.
Processed 17% -> 100 out of 600 rows.
Processed 100% -> 600 out of 600 rows.
Processed 267% -> 1600 out of 600 rows.
Processed 33% -> 200 out of 600 rows.
Processed 217% -> 1300 out of 600 rows.
Processed 233% -> 1400 out of 600 rows.
Average Pre-Fine-Tuning Similarity: 0.8459771


## Prepare Data for Fine-Tuning
Define a dataset class to tokenize query and news_content for the model.

In [12]:
from torch.utils.data import Dataset, DataLoader

class QueryNewsDataset(Dataset):
    def __init__(self, df, tokenizer, max_length=128):
        self.queries = df['query'].tolist()
        self.news = df['news_content'].tolist()
        self.tokenizer = tokenizer
        self.max_length = max_length
    
    def __len__(self):
        return len(self.queries)
    
    def __getitem__(self, idx):
        query_text = self.queries[idx]
        news_text = self.news[idx]
        
        query_inputs = self.tokenizer(query_text, padding='max_length', truncation=True, max_length=self.max_length, return_tensors="pt")
        news_inputs = self.tokenizer(news_text, padding='max_length', truncation=True, max_length=self.max_length, return_tensors="pt")
        
        return {
            'query_input_ids': query_inputs['input_ids'].squeeze(0),
            'query_attention_mask': query_inputs['attention_mask'].squeeze(0),
            'news_input_ids': news_inputs['input_ids'].squeeze(0),
            'news_attention_mask': news_inputs['attention_mask'].squeeze(0)
        }

# Create DataLoader for fine-tuning
train_loader = DataLoader(QueryNewsDataset(test_df, tokenizer), batch_size=8, shuffle=True)

## Fine-Tune the Model with QLoRA

Define a cosine similarity loss and run the fine-tuning loop.

In [13]:
import torch
from torch.nn import functional as F
from transformers import AdamW

# Optimizer
optimizer = AdamW(model.parameters(), lr=2e-5)

# Custom similarity loss function
def similarity_loss(embedding1, embedding2):
    cos_sim = F.cosine_similarity(embedding1, embedding2)
    return 1 - cos_sim.mean()

# Fine-tuning loop with progress updates
model.train()
for epoch in range(3):  # Adjust the number of epochs as needed
    total_loss = 0
    for i, batch in enumerate(train_loader):
        optimizer.zero_grad()
        
        # Query embeddings
        query_embeddings = model(
            input_ids=batch['query_input_ids'],
            attention_mask=batch['query_attention_mask']
        ).last_hidden_state.mean(dim=1)
        
        # News content embeddings
        news_embeddings = model(
            input_ids=batch['news_input_ids'],
            attention_mask=batch['news_attention_mask']
        ).last_hidden_state.mean(dim=1)
        
        # Calculate similarity loss and optimize
        loss = similarity_loss(query_embeddings, news_embeddings)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        
        # Print progress within each epoch
        if (i + 1) % 10 == 0:
            percent_complete = ((i + 1) / len(train_loader)) * 100
            print(f"Epoch {epoch + 1} - Processed {percent_complete:.0f}% -> {i + 1} out of {len(train_loader)} batches.")

    avg_loss = total_loss / len(train_loader)
    print(f"Epoch {epoch + 1} completed with average loss: {avg_loss:.4f}")


/Users/who/projects/task1/venv/lib/python3.11/site-packages/transformers/optimization.py:591: FutureWarning: This implementation of AdamW is deprecated and will be removed in a future version. Use the PyTorch implementation torch.optim.AdamW instead, or set `no_deprecation_warning=True` to disable this warning
  warnings.warn(


Epoch 1 - Processed 13% -> 10 out of 75 batches.
Epoch 1 - Processed 27% -> 20 out of 75 batches.
Epoch 1 - Processed 40% -> 30 out of 75 batches.
Epoch 1 - Processed 53% -> 40 out of 75 batches.
Epoch 1 - Processed 67% -> 50 out of 75 batches.
Epoch 1 - Processed 80% -> 60 out of 75 batches.
Epoch 1 - Processed 93% -> 70 out of 75 batches.
Epoch 1 completed with average loss: 0.1344
Epoch 2 - Processed 13% -> 10 out of 75 batches.
Epoch 2 - Processed 27% -> 20 out of 75 batches.
Epoch 2 - Processed 40% -> 30 out of 75 batches.
Epoch 2 - Processed 53% -> 40 out of 75 batches.
Epoch 2 - Processed 67% -> 50 out of 75 batches.
Epoch 2 - Processed 80% -> 60 out of 75 batches.
Epoch 2 - Processed 93% -> 70 out of 75 batches.
Epoch 2 completed with average loss: 0.1058
Epoch 3 - Processed 13% -> 10 out of 75 batches.
Epoch 3 - Processed 27% -> 20 out of 75 batches.
Epoch 3 - Processed 40% -> 30 out of 75 batches.
Epoch 3 - Processed 53% -> 40 out of 75 batches.
Epoch 3 - Processed 67% -> 50 

## Post-Fine-Tuning Performance Evaluation
Recompute the cosine similarity for query and new_content pairs after fine-tuning.

In [14]:
# Post-fine-tuning similarity calculation with progress updates
similarities_after = []
model.eval()
for i, row in test_df.iterrows():
    query_embedding = get_embeddings(row['query'], model, tokenizer)
    content_embedding = get_embeddings(row['news_content'], model, tokenizer)
    similarity = cosine_similarity(query_embedding, content_embedding)[0][0]
    similarities_after.append(similarity)
    
    # Print progress
    if (i + 1) % 100 == 0:
        percent_complete = ((i + 1) / len(test_df)) * 100
        print(f"Processed {percent_complete:.0f}% -> {i + 1} out of {len(test_df)} rows.")

test_df['similarity_after'] = similarities_after
print("Average Post-Fine-Tuning Similarity:", sum(similarities_after) / len(similarities_after))

Processed 300% -> 1800 out of 600 rows.
Processed 17% -> 100 out of 600 rows.
Processed 100% -> 600 out of 600 rows.
Processed 267% -> 1600 out of 600 rows.
Processed 33% -> 200 out of 600 rows.
Processed 217% -> 1300 out of 600 rows.
Processed 233% -> 1400 out of 600 rows.
Average Post-Fine-Tuning Similarity: 0.94892925


## Save Results and compare

In [15]:
# Save the comparison results to CSV
test_df.to_csv('fine_tuning_results_test_set.csv', index=False)
print("Results saved to fine_tuning_results_test_set.csv")

# Display average similarity improvement
avg_similarity_before = test_df['similarity_before'].mean()
avg_similarity_after = test_df['similarity_after'].mean()
print(f"Average Similarity Before Fine-Tuning: {avg_similarity_before:.4f}")
print(f"Average Similarity After Fine-Tuning: {avg_similarity_after:.4f}")
print(f"Improvement: {avg_similarity_after - avg_similarity_before:.4f}")

Results saved to fine_tuning_results_test_set.csv
Average Similarity Before Fine-Tuning: 0.8460
Average Similarity After Fine-Tuning: 0.9489
Improvement: 0.1030


##  Save the Model and Tokenizer

In [16]:
from transformers import AutoModel

# Paths for saving base model and LoRA adapter
base_model_directory = "../models/fine_tuned_e5_large_v2_lora/base_model"
adapter_directory = "../models/fine_tuned_e5_large_v2_lora/adapter"

# Ensure the directories exist
import os
os.makedirs(base_model_directory, exist_ok=True)
os.makedirs(adapter_directory, exist_ok=True)

# Save the base model separately
base_model = AutoModel.from_pretrained("intfloat/e5-large-v2")
base_model.save_pretrained(base_model_directory)

# Save the adapter (fine-tuned LoRA weights)
model.save_pretrained(adapter_directory)

# Save the tokenizer in the main directory
tokenizer.save_pretrained("../models/fine_tuned_e5_large_v2_lora")

print(f"Base model saved to {base_model_directory}")
print(f"Adapter (fine-tuned model) saved to {adapter_directory}")
print("Tokenizer saved successfully.")


Base model saved to ../models/fine_tuned_e5_large_v2_lora/base_model
Adapter (fine-tuned model) saved to ../models/fine_tuned_e5_large_v2_lora/adapter
Tokenizer saved successfully.


In [17]:
# from transformers import AutoTokenizer

# # Define a path to save the model and tokenizer
# save_directory = "../models/fine_tuned_e5_large_v2_lora"

# # Save the fine-tuned model
# model.save_pretrained(save_directory)

# # Save the tokenizer
# tokenizer.save_pretrained(save_directory)

# print(f"Model and tokenizer saved to {save_directory}")

## Try the model on the news data set with more general queries

In [18]:
# Generate embeddings for all news items and store in 'embedding' column
df['e5_embedding'] = df['content'].apply(lambda x: get_embeddings(x, model, tokenizer))
df

,_id,title,pubDate,guid,link,description,content,openai_embedding,e5_embedding_before_tuning,e5_embedding
0,671e2d0aeccd4a5f4883e981,Justin Welby: Political leaders should treat o...,2024-01-01 00:00:04,https://www.bbc.co.uk/news/uk-67844356,https://www.bbc.co.uk/news/uk-67844356?at_medi...,The Archbishop of Canterbury urges politicians...,Justin Welby: Political leaders should treat o...,"[0.02301846444606781, -0.006470758933573961, 0...","[[0.51037025, -0.79349065, -0.13040037, -0.139...","[[0.3661322, -1.1412252, 0.17336315, -0.137644..."
1,671e2d0aeccd4a5f4883e982,Almost three million tested for cancer in England,2024-01-01 00:09:56,https://www.bbc.co.uk/news/health-67841348,https://www.bbc.co.uk/news/health-67841348?at_...,Record numbers are being tested for cancer but...,Almost three million tested for cancer in Engl...,"[0.0134049067273736, 0.012411133386194706, 0.0...","[[0.3748049, -1.0251585, -0.13934056, -0.46842...","[[0.25168958, -1.3728334, 0.1894472, -0.401590..."
2,671e2d0aeccd4a5f4883e983,Household energy price rise of 5% comes into f...,2024-01-01 00:00:16,https://www.bbc.co.uk/news/business-67785266,https://www.bbc.co.uk/news/business-67785266?a...,A higher cap for the next three months adds £9...,Household energy price rise of 5% comes into f...,"[-0.00222824071533978, 0.032821644097566605, 0...","[[0.5421233, -0.91381705, 0.37531197, -0.54616...","[[0.28375754, -1.2104545, 0.43982285, -0.32502..."
3,671e2d0aeccd4a5f4883e984,Primrose Hill stabbing: Harry Pitman named as ...,2024-01-01 17:11:13,https://www.bbc.co.uk/news/uk-england-london-6...,https://www.bbc.co.uk/news/uk-england-london-6...,"Harry Pitman, 16, was attacked on London's Pri...",Primrose Hill stabbing: Harry Pitman named as ...,"[-0.0027212619315832853, 0.003952309023588896,...","[[-0.32640272, -1.4084414, -0.064704806, -0.96...","[[-0.13805056, -1.6102695, 0.20174393, -0.8064..."
4,671e2d0aeccd4a5f4883e985,Israel Supreme Court strikes down judicial ref...,2024-01-01 19:47:58,https://www.bbc.co.uk/news/world-middle-east-6...,https://www.bbc.co.uk/news/world-middle-east-6...,The controversial plans triggered nationwide p...,Israel Supreme Court strikes down judicial ref...,"[0.026911897584795952, 0.025074539706110954, 0...","[[-0.28679425, -0.8542533, 0.63113433, -0.7137...","[[-0.12946197, -1.1774564, 0.642205, -0.448401..."
...,...,...,...,...,...,...,...,...,...,...
7255,671e2d0aeccd4a5f488405d8,Aer Lingus pilots stage eight-hour work stoppage,2024-06-29 13:49:48,https://www.bbc.com/news/articles/cpv3dg31x75o#12,https://www.bbc.com/news/articles/cpv3dg31x75o,"The strike action, which lasted for eight hour...",Aer Lingus pilots stage eight-hour work stoppa...,"[-0.03716661408543587, 0.040872685611248016, 0...","[[0.2114126, -0.6251888, -0.22534753, -0.19603...","[[0.22619294, -1.0267925, -0.01779555, -0.2222..."
7256,671e2d0aeccd4a5f488405d9,Another council sets up emergency postal vote ...,2024-06-29 12:43:00,https://www.bbc.com/news/articles/cx02knj0l7xo#12,https://www.bbc.com/news/articles/cx02knj0l7xo,Replacement voting packs are issued in some ar...,Another council sets up emergency postal vote ...,"[0.020275646820664406, 0.06969498842954636, 0....","[[-0.3025538, -1.7177178, 0.15702304, -0.68241...","[[-0.13594653, -1.744225, 0.29958287, -0.44506..."
7257,671e2d0aeccd4a5f488405da,"Politicians forget we're voters, says Gypsy woman",2024-06-29 21:00:01,https://www.bbc.com/news/articles/c3ggvyrk0k6o#12,https://www.bbc.com/news/articles/c3ggvyrk0k6o,A campaign group says Travellers are treated l...,"Politicians forget we're voters, says Gypsy wo...","[0.002537588821724057, 0.060002345591783524, -...","[[0.18100068, -1.0736654, 0.33410802, -0.26385...","[[0.15298942, -1.3693814, 0.47441474, -0.19119..."
7258,671e2d0aeccd4a5f488405db,Where are the seats that could decide the elec...,2024-06-25 14:49:05,https://www.bbc.com/news/articles/c133p016pg4o#1,https://www.bbc.com/news/articles/c133p016pg4o,The parties' top battleground targets across t

In [19]:
def find_top_news(query, df, model, tokenizer, top_k=10):
    query_embedding = get_embeddings(query, model, tokenizer)

    # Calculate cosine similarity between query and all news embeddings
    similarities = df['e5_embedding'].apply(lambda x: cosine_similarity(query_embedding, x)[0][0])
    
    # Add similarity scores to the DataFrame
    df['e5_similarity'] = similarities
    
    # Sort by similarity and return the top_k results
    top_matches = df.nlargest(top_k, 'e5_similarity')
    
    # Clean up by dropping the 'similarity' column
    df.drop(columns=['e5_similarity'], inplace=True)
    
    return top_matches[['content', 'e5_similarity']]

In [20]:
"""

"""
test_queries = ["Products for Color-blind people","JEEP off-road vehicles","What is the origin of Bamba snack","Recommended diet and exercise habits for managing obesity","Keep your children safe on the internet","What are the lates news about AI ethics?","How the war between Israel and Hamas is affecting global politics?","Latest trends in the fashion industry","Interest rates","biden vs trump","Best Marvel movie this year","Northern Lights"]
for user_query in test_queries:
    print(f"\n\nQuery: {user_query}")
    top_news = find_top_news(user_query, df, model, tokenizer)
    for _, row in top_news.iterrows():
                print(f"\033[96m Similarity: {row['e5_similarity']:.3f}::\033[0m {row['content']}")



Query: Products for Color-blind people
 Similarity: 0.907:: Ethnic bias in health devices 'puts patients at risk'. The review focuses on devices frequently used in the NHS, like pulse oximeters and skin cancer apps.
 Similarity: 0.903:: 'I’ve invented an alternative to uncomfortable smear tests'. Sânziana Foia's non-invasive test detects harmful virus strains and delivers fast results at home.
 Similarity: 0.901:: What is behind the TikTok thirst for Stanley water cups?. People are camping outside supermarkets in the US to try to buy the latest internet on-trend item.
 Similarity: 0.901:: Top sunscreens fail protection tests, Which? says. Some cheaper lotions from supermarkets Aldi and Lidl outperformed more expensive brands, Which? said.
 Similarity: 0.900:: Where workers are exploited to harvest an everyday ingredient. Brazilian workers face degrading conditions to harvest palm wax used in sweets, pills and lipstick.
 Similarity: 0.899:: How portable X-ray machines are helping remo

## Loading the Fine-Tuned Model for Embeddings in Your App

In [21]:
from transformers import AutoTokenizer, AutoModel

save_directory = "../models/fine_tuned_e5_large_v2_lora"

# Load the fine-tuned model and tokenizer
model = AutoModel.from_pretrained(save_directory)
tokenizer = AutoTokenizer.from_pretrained(save_directory)

# Now you can use `model` for embeddings in your app

ValueError: Unrecognized model in ../models/fine_tuned_e5_large_v2_lora. Should have a `model_type` key in its config.json, or contain one of the following strings in its name: albert, align, altclip, audio-spectrogram-transformer, autoformer, bark, bart, beit, bert, bert-generation, big_bird, bigbird_pegasus, biogpt, bit, blenderbot, blenderbot-small, blip, blip-2, bloom, bridgetower, bros, camembert, canine, chameleon, chinese_clip, chinese_clip_vision_model, clap, clip, clip_text_model, clip_vision_model, clipseg, clvp, code_llama, codegen, cohere, conditional_detr, convbert, convnext, convnextv2, cpmant, ctrl, cvt, dac, data2vec-audio, data2vec-text, data2vec-vision, dbrx, deberta, deberta-v2, decision_transformer, deformable_detr, deit, depth_anything, deta, detr, dinat, dinov2, distilbert, donut-swin, dpr, dpt, efficientformer, efficientnet, electra, encodec, encoder-decoder, ernie, ernie_m, esm, falcon, falcon_mamba, fastspeech2_conformer, flaubert, flava, fnet, focalnet, fsmt, 

## Generate Embeddings with the Loaded Model

In [11]:
import torch

def get_embeddings(text, model, tokenizer):
    inputs = tokenizer(text, return_tensors="pt", padding=True, truncation=True)
    with torch.no_grad():
        embeddings = model(**inputs).last_hidden_state.mean(dim=1).cpu().numpy()
    return embeddings

# Example usage
text = "Example query for embedding generation"
embedding = get_embeddings(text, model, tokenizer)
embedding

# Fine-tuning using less data

In [3]:
import pandas as pd
from sklearn.model_selection import train_test_split

from pymongo import MongoClient
import pandas as pd

# MongoDB connection details
mongo_uri = "mongodb://localhost:27017/"
db_name = "task1"
collection_name = "bbc_news"

# Connect to MongoDB
client = MongoClient(mongo_uri)
db = client[db_name]
collection = db[collection_name]

documents = list(collection.find())
df = pd.DataFrame(documents)
client.close()

queries_df = pd.read_csv('notebooks_data/generated_queries.csv')
queries_df = queries_df.iloc[:1000]

train_df, test_df = train_test_split(queries_df, test_size=0.3, random_state=42)
print(f"Training set size: {len(train_df)}")
print(f"Test set size: {len(test_df)}")

from transformers import AutoTokenizer, AutoModel
from peft import LoraConfig, get_peft_model

# Load tokenizer and model without quantization
model_name = "intfloat/e5-large-v2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)  # Standard loading, no quantization

# Define QLoRA configuration
lora_config = LoraConfig(
    r=8,               # Rank of the low-rank matrices
    lora_alpha=8,      # Scaling factor for LoRA updates
    target_modules=["key", "query", "value"],  # Adjust based on model structure if necessary
    lora_dropout=0.1,   # Dropout for LoRA layers
    bias="none"         # Exclude biases to focus on core layers
)

# Wrap the model with QLoRA
model = get_peft_model(model, lora_config)

Training set size: 700
Test set size: 300


In [4]:
import torch

def get_embeddings(text, model, tokenizer):
    inputs = tokenizer(text, return_tensors="pt", padding=True, truncation=True).to("cpu")
    with torch.no_grad():
        embeddings = model(**inputs).last_hidden_state.mean(dim=1).cpu().numpy()
    return embeddings


df['e5_embedding_before_tuning'] = df['content'].apply(lambda x: get_embeddings(x, model, tokenizer))

from sklearn.metrics.pairwise import cosine_similarity

def find_top_news(query, df, model, tokenizer, top_k=10):
    query_embedding = get_embeddings(query, model, tokenizer)

    # Calculate cosine similarity between query and all news embeddings
    similarities = df['e5_embedding_before_tuning'].apply(lambda x: cosine_similarity(query_embedding, x)[0][0])
    
    # Add similarity scores to the DataFrame
    df['e5_similarity'] = similarities
    
    # Sort by similarity and return the top_k results
    top_matches = df.nlargest(top_k, 'e5_similarity')
    
    # Clean up by dropping the 'similarity' column
    df.drop(columns=['e5_similarity'], inplace=True)
    
    return top_matches[['content', 'e5_similarity']]

test_queries = ["Products for Color-blind people","JEEP off-road vehicles","What is the origin of Bamba snack","Recommended diet and exercise habits for managing obesity","Keep your children safe on the internet","What are the lates news about AI ethics?","How the war between Israel and Hamas is affecting global politics?","Latest trends in the fashion industry","Interest rates","biden vs trump","Best Marvel movie this year","Northern Lights"]
for user_query in test_queries:
    print(f"\n\nQuery: {user_query}")
    top_news = find_top_news(user_query, df, model, tokenizer)
    for _, row in top_news.iterrows():
                print(f"\033[96m Similarity: {row['e5_similarity']:.3f}::\033[0m {row['content']}")



Query: Products for Color-blind people
 Similarity: 0.785:: Cost of living: How can I make extra income?. Some people have been earning thousands selling clothes and other items since they were in school.
 Similarity: 0.780:: Ethnic bias in health devices 'puts patients at risk'. The review focuses on devices frequently used in the NHS, like pulse oximeters and skin cancer apps.
 Similarity: 0.774:: Americanswers! What is Project 2025? Will Biden take a cognitive test?. We answer your questions on the biggest issues in the run-up to the election
 Similarity: 0.774:: Electioncast: Laura Kuenssberg and Adam Fleming answer your questions. Adam and Laura answer your election questions
 Similarity: 0.773:: The List. Exploring 9 amazing space missions.
 Similarity: 0.773:: Who can I vote for in the general election?. Find out which constituency you are in, who you can vote for and where you can vote using our postcode search.
 Similarity: 0.773:: 'I’ve invented an alternative to uncomforta

In [5]:
from sklearn.metrics.pairwise import cosine_similarity

# Helper function to get embeddings
def get_embeddings(text, model, tokenizer):
    inputs = tokenizer(text, return_tensors="pt", padding=True, truncation=True).to("cpu")
    with torch.no_grad():
        embeddings = model(**inputs).last_hidden_state.mean(dim=1).cpu().numpy()
    return embeddings

# Calculate pre-fine-tuning similarities
similarities_before = []
for index, row in test_df.iterrows():
    query_embedding = get_embeddings(row['query'], model, tokenizer)
    content_embedding = get_embeddings(row['news_content'], model, tokenizer)
    similarity = cosine_similarity(query_embedding, content_embedding)[0][0]
    similarities_before.append(similarity)

    # Print the progress
    if (index + 1) % 100 == 0:  # Adjust the modulus value to print more or less frequently
        percent_complete = ((index + 1) / len(test_df)) * 100
        print(f"Processed {percent_complete:.0f}% -> {index + 1} out of {len(test_df)} rows.")

# Store pre-fine-tuning similarities
test_df['similarity_before'] = similarities_before
test_df.to_csv('notebooks_data/generated_queries_with_similarity_before.csv', index=False)
print("Average Pre-Fine-Tuning Similarity:", sum(similarities_before) / len(similarities_before))

Processed 300% -> 900 out of 300 rows.
Processed 167% -> 500 out of 300 rows.
Processed 100% -> 300 out of 300 rows.
Processed 200% -> 600 out of 300 rows.
Processed 67% -> 200 out of 300 rows.
Processed 267% -> 800 out of 300 rows.
Average Pre-Fine-Tuning Similarity: 0.8498281


In [6]:
from torch.utils.data import Dataset, DataLoader

class QueryNewsDataset(Dataset):
    def __init__(self, df, tokenizer, max_length=128):
        self.queries = df['query'].tolist()
        self.news = df['news_content'].tolist()
        self.tokenizer = tokenizer
        self.max_length = max_length
    
    def __len__(self):
        return len(self.queries)
    
    def __getitem__(self, idx):
        query_text = self.queries[idx]
        news_text = self.news[idx]
        
        query_inputs = self.tokenizer(query_text, padding='max_length', truncation=True, max_length=self.max_length, return_tensors="pt")
        news_inputs = self.tokenizer(news_text, padding='max_length', truncation=True, max_length=self.max_length, return_tensors="pt")
        
        return {
            'query_input_ids': query_inputs['input_ids'].squeeze(0),
            'query_attention_mask': query_inputs['attention_mask'].squeeze(0),
            'news_input_ids': news_inputs['input_ids'].squeeze(0),
            'news_attention_mask': news_inputs['attention_mask'].squeeze(0)
        }

# Create DataLoader for fine-tuning
train_loader = DataLoader(QueryNewsDataset(test_df, tokenizer), batch_size=8, shuffle=True)




import torch
from torch.nn import functional as F
from transformers import AdamW

# Optimizer
optimizer = AdamW(model.parameters(), lr=2e-5)

# Custom similarity loss function
def similarity_loss(embedding1, embedding2):
    cos_sim = F.cosine_similarity(embedding1, embedding2)
    return 1 - cos_sim.mean()

# Fine-tuning loop with progress updates
model.train()
for epoch in range(3):  # Adjust the number of epochs as needed
    total_loss = 0
    for i, batch in enumerate(train_loader):
        optimizer.zero_grad()
        
        # Query embeddings
        query_embeddings = model(
            input_ids=batch['query_input_ids'],
            attention_mask=batch['query_attention_mask']
        ).last_hidden_state.mean(dim=1)
        
        # News content embeddings
        news_embeddings = model(
            input_ids=batch['news_input_ids'],
            attention_mask=batch['news_attention_mask']
        ).last_hidden_state.mean(dim=1)
        
        # Calculate similarity loss and optimize
        loss = similarity_loss(query_embeddings, news_embeddings)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        
        # Print progress within each epoch
        if (i + 1) % 10 == 0:
            percent_complete = ((i + 1) / len(train_loader)) * 100
            print(f"Epoch {epoch + 1} - Processed {percent_complete:.0f}% -> {i + 1} out of {len(train_loader)} batches.")

    avg_loss = total_loss / len(train_loader)
    print(f"Epoch {epoch + 1} completed with average loss: {avg_loss:.4f}")

/Users/who/projects/task1/venv/lib/python3.11/site-packages/transformers/optimization.py:591: FutureWarning: This implementation of AdamW is deprecated and will be removed in a future version. Use the PyTorch implementation torch.optim.AdamW instead, or set `no_deprecation_warning=True` to disable this warning
  warnings.warn(


Epoch 1 - Processed 26% -> 10 out of 38 batches.
Epoch 1 - Processed 53% -> 20 out of 38 batches.
Epoch 1 - Processed 79% -> 30 out of 38 batches.
Epoch 1 completed with average loss: 0.1364
Epoch 2 - Processed 26% -> 10 out of 38 batches.
Epoch 2 - Processed 53% -> 20 out of 38 batches.
Epoch 2 - Processed 79% -> 30 out of 38 batches.
Epoch 2 completed with average loss: 0.1276
Epoch 3 - Processed 26% -> 10 out of 38 batches.
Epoch 3 - Processed 53% -> 20 out of 38 batches.
Epoch 3 - Processed 79% -> 30 out of 38 batches.
Epoch 3 completed with average loss: 0.1138


In [7]:
# Post-fine-tuning similarity calculation with progress updates
similarities_after = []
model.eval()
for i, row in test_df.iterrows():
    query_embedding = get_embeddings(row['query'], model, tokenizer)
    content_embedding = get_embeddings(row['news_content'], model, tokenizer)
    similarity = cosine_similarity(query_embedding, content_embedding)[0][0]
    similarities_after.append(similarity)
    
    # Print progress
    if (i + 1) % 100 == 0:
        percent_complete = ((i + 1) / len(test_df)) * 100
        print(f"Processed {percent_complete:.0f}% -> {i + 1} out of {len(test_df)} rows.")

test_df['similarity_after'] = similarities_after
print("Average Post-Fine-Tuning Similarity:", sum(similarities_after) / len(similarities_after))

# Display average similarity improvement
avg_similarity_before = test_df['similarity_before'].mean()
avg_similarity_after = test_df['similarity_after'].mean()
print(f"Average Similarity Before Fine-Tuning: {avg_similarity_before:.4f}")
print(f"Average Similarity After Fine-Tuning: {avg_similarity_after:.4f}")
print(f"Improvement: {avg_similarity_after - avg_similarity_before:.4f}")


test_queries = ["Products for Color-blind people","JEEP off-road vehicles","What is the origin of Bamba snack","Recommended diet and exercise habits for managing obesity","Keep your children safe on the internet","What are the lates news about AI ethics?","How the war between Israel and Hamas is affecting global politics?","Latest trends in the fashion industry","Interest rates","biden vs trump","Best Marvel movie this year","Northern Lights"]
for user_query in test_queries:
    print(f"\n\nQuery: {user_query}")
    top_news = find_top_news(user_query, df, model, tokenizer)
    for _, row in top_news.iterrows():
                print(f"\033[96m Similarity: {row['e5_similarity']:.3f}::\033[0m {row['content']}")

Processed 300% -> 900 out of 300 rows.
Processed 167% -> 500 out of 300 rows.
Processed 100% -> 300 out of 300 rows.
Processed 200% -> 600 out of 300 rows.
Processed 67% -> 200 out of 300 rows.
Processed 267% -> 800 out of 300 rows.
Average Post-Fine-Tuning Similarity: 0.8846712
Average Similarity Before Fine-Tuning: 0.8498
Average Similarity After Fine-Tuning: 0.8847
Improvement: 0.0348


Query: Products for Color-blind people
 Similarity: 0.801:: Cost of living: How can I make extra income?. Some people have been earning thousands selling clothes and other items since they were in school.
 Similarity: 0.798:: Ethnic bias in health devices 'puts patients at risk'. The review focuses on devices frequently used in the NHS, like pulse oximeters and skin cancer apps.
 Similarity: 0.793:: Electioncast: Laura Kuenssberg and Adam Fleming answer your questions. Adam and Laura answer your election questions
 Similarity: 0.791:: Aphantasia: Why I cannot see my children in my mind. Not everyone 

# Fine tuning Rank 16

In [8]:
import pandas as pd
from sklearn.model_selection import train_test_split

from pymongo import MongoClient
import pandas as pd

# MongoDB connection details
mongo_uri = "mongodb://localhost:27017/"
db_name = "task1"
collection_name = "bbc_news"

# Connect to MongoDB
client = MongoClient(mongo_uri)
db = client[db_name]
collection = db[collection_name]

documents = list(collection.find())
df = pd.DataFrame(documents)
client.close()

queries_df = pd.read_csv('notebooks_data/generated_queries.csv')

train_df, test_df = train_test_split(queries_df, test_size=0.3, random_state=42)
print(f"Training set size: {len(train_df)}")
print(f"Test set size: {len(test_df)}")

from transformers import AutoTokenizer, AutoModel
from peft import LoraConfig, get_peft_model

# Load tokenizer and model without quantization
model_name = "intfloat/e5-large-v2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)  # Standard loading, no quantization

# Define QLoRA configuration
lora_config = LoraConfig(
    r=16,               # Rank of the low-rank matrices
    lora_alpha=32,      # Scaling factor for LoRA updates
    target_modules=["key", "query", "value"],  # Adjust based on model structure if necessary
    lora_dropout=0.1,   # Dropout for LoRA layers
    bias="none"         # Exclude biases to focus on core layers
)

# Wrap the model with QLoRA
model = get_peft_model(model, lora_config)

Training set size: 1399
Test set size: 600


In [9]:
import torch

def get_embeddings(text, model, tokenizer):
    inputs = tokenizer(text, return_tensors="pt", padding=True, truncation=True).to("cpu")
    with torch.no_grad():
        embeddings = model(**inputs).last_hidden_state.mean(dim=1).cpu().numpy()
    return embeddings


from sklearn.metrics.pairwise import cosine_similarity

def find_top_news(query, df, model, tokenizer, top_k=10):
    query_embedding = get_embeddings(query, model, tokenizer)

    # Calculate cosine similarity between query and all news embeddings
    similarities = df['e5_embedding_before_tuning'].apply(lambda x: cosine_similarity(query_embedding, x)[0][0])
    
    # Add similarity scores to the DataFrame
    df['e5_similarity'] = similarities
    
    # Sort by similarity and return the top_k results
    top_matches = df.nlargest(top_k, 'e5_similarity')
    
    # Clean up by dropping the 'similarity' column
    df.drop(columns=['e5_similarity'], inplace=True)
    
    return top_matches[['content', 'e5_similarity']]


In [10]:
from torch.utils.data import Dataset, DataLoader

class QueryNewsDataset(Dataset):
    def __init__(self, df, tokenizer, max_length=128):
        self.queries = df['query'].tolist()
        self.news = df['news_content'].tolist()
        self.tokenizer = tokenizer
        self.max_length = max_length
    
    def __len__(self):
        return len(self.queries)
    
    def __getitem__(self, idx):
        query_text = self.queries[idx]
        news_text = self.news[idx]
        
        query_inputs = self.tokenizer(query_text, padding='max_length', truncation=True, max_length=self.max_length, return_tensors="pt")
        news_inputs = self.tokenizer(news_text, padding='max_length', truncation=True, max_length=self.max_length, return_tensors="pt")
        
        return {
            'query_input_ids': query_inputs['input_ids'].squeeze(0),
            'query_attention_mask': query_inputs['attention_mask'].squeeze(0),
            'news_input_ids': news_inputs['input_ids'].squeeze(0),
            'news_attention_mask': news_inputs['attention_mask'].squeeze(0)
        }

# Create DataLoader for fine-tuning
train_loader = DataLoader(QueryNewsDataset(test_df, tokenizer), batch_size=8, shuffle=True)




import torch
from torch.nn import functional as F
from transformers import AdamW

# Optimizer
optimizer = AdamW(model.parameters(), lr=2e-5)

# Custom similarity loss function
def similarity_loss(embedding1, embedding2):
    cos_sim = F.cosine_similarity(embedding1, embedding2)
    return 1 - cos_sim.mean()

# Fine-tuning loop with progress updates
model.train()
for epoch in range(3):  # Adjust the number of epochs as needed
    total_loss = 0
    for i, batch in enumerate(train_loader):
        optimizer.zero_grad()
        
        # Query embeddings
        query_embeddings = model(
            input_ids=batch['query_input_ids'],
            attention_mask=batch['query_attention_mask']
        ).last_hidden_state.mean(dim=1)
        
        # News content embeddings
        news_embeddings = model(
            input_ids=batch['news_input_ids'],
            attention_mask=batch['news_attention_mask']
        ).last_hidden_state.mean(dim=1)
        
        # Calculate similarity loss and optimize
        loss = similarity_loss(query_embeddings, news_embeddings)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        
        # Print progress within each epoch
        if (i + 1) % 10 == 0:
            percent_complete = ((i + 1) / len(train_loader)) * 100
            print(f"Epoch {epoch + 1} - Processed {percent_complete:.0f}% -> {i + 1} out of {len(train_loader)} batches.")

    avg_loss = total_loss / len(train_loader)
    print(f"Epoch {epoch + 1} completed with average loss: {avg_loss:.4f}")

/Users/who/projects/task1/venv/lib/python3.11/site-packages/transformers/optimization.py:591: FutureWarning: This implementation of AdamW is deprecated and will be removed in a future version. Use the PyTorch implementation torch.optim.AdamW instead, or set `no_deprecation_warning=True` to disable this warning
  warnings.warn(


Epoch 1 - Processed 13% -> 10 out of 75 batches.
Epoch 1 - Processed 27% -> 20 out of 75 batches.
Epoch 1 - Processed 40% -> 30 out of 75 batches.
Epoch 1 - Processed 53% -> 40 out of 75 batches.
Epoch 1 - Processed 67% -> 50 out of 75 batches.
Epoch 1 - Processed 80% -> 60 out of 75 batches.
Epoch 1 - Processed 93% -> 70 out of 75 batches.
Epoch 1 completed with average loss: 0.1149
Epoch 2 - Processed 13% -> 10 out of 75 batches.
Epoch 2 - Processed 27% -> 20 out of 75 batches.
Epoch 2 - Processed 40% -> 30 out of 75 batches.
Epoch 2 - Processed 53% -> 40 out of 75 batches.
Epoch 2 - Processed 67% -> 50 out of 75 batches.
Epoch 2 - Processed 80% -> 60 out of 75 batches.
Epoch 2 - Processed 93% -> 70 out of 75 batches.
Epoch 2 completed with average loss: 0.0459
Epoch 3 - Processed 13% -> 10 out of 75 batches.
Epoch 3 - Processed 27% -> 20 out of 75 batches.
Epoch 3 - Processed 40% -> 30 out of 75 batches.
Epoch 3 - Processed 53% -> 40 out of 75 batches.
Epoch 3 - Processed 67% -> 50 

In [12]:
# Post-fine-tuning similarity calculation with progress updates
similarities_after = []
model.eval()
for i, row in test_df.iterrows():
    query_embedding = get_embeddings(row['query'], model, tokenizer)
    content_embedding = get_embeddings(row['news_content'], model, tokenizer)
    similarity = cosine_similarity(query_embedding, content_embedding)[0][0]
    similarities_after.append(similarity)
    
    # Print progress
    if (i + 1) % 100 == 0:
        percent_complete = ((i + 1) / len(test_df)) * 100
        print(f"Processed {percent_complete:.0f}% -> {i + 1} out of {len(test_df)} rows.")

print("Average Post-Fine-Tuning Similarity:", sum(similarities_after) / len(similarities_after))

df['e5_embedding_before_tuning'] = df['content'].apply(lambda x: get_embeddings(x, model, tokenizer))

test_queries = ["Products for Color-blind people","JEEP off-road vehicles","What is the origin of Bamba snack","Recommended diet and exercise habits for managing obesity","Keep your children safe on the internet","What are the lates news about AI ethics?","How the war between Israel and Hamas is affecting global politics?","Latest trends in the fashion industry","Interest rates","biden vs trump","Best Marvel movie this year","Northern Lights"]
for user_query in test_queries:
    print(f"\n\nQuery: {user_query}")
    top_news = find_top_news(user_query, df, model, tokenizer)
    for _, row in top_news.iterrows():
                print(f"\033[96m Similarity: {row['e5_similarity']:.3f}::\033[0m {row['content']}")

Processed 300% -> 1800 out of 600 rows.
Processed 17% -> 100 out of 600 rows.
Processed 100% -> 600 out of 600 rows.
Processed 267% -> 1600 out of 600 rows.
Processed 33% -> 200 out of 600 rows.
Processed 217% -> 1300 out of 600 rows.
Processed 233% -> 1400 out of 600 rows.
Average Post-Fine-Tuning Similarity: 0.9971273


Query: Products for Color-blind people
 Similarity: 0.993:: Where workers are exploited to harvest an everyday ingredient. Brazilian workers face degrading conditions to harvest palm wax used in sweets, pills and lipstick.
 Similarity: 0.992:: Seven conditions that your local chemist can now treat. Pharmacies in England are able to supply antibiotics and advice without patients seeing a doctor.
 Similarity: 0.992:: A wine a mile: Marathon runner tastes 25 glasses. Tom Gilbey sampled 25 wines in a blind taste test challenge correctly identifying 21.
 Similarity: 0.992:: Removing large wine glasses from sale cuts drinking - study. With the largest measure off the menu, 

# 1K rank 4 alpha 16

In [16]:
import pandas as pd
from sklearn.model_selection import train_test_split

from pymongo import MongoClient
import pandas as pd

# MongoDB connection details
mongo_uri = "mongodb://localhost:27017/"
db_name = "task1"
collection_name = "bbc_news"

# Connect to MongoDB
client = MongoClient(mongo_uri)
db = client[db_name]
collection = db[collection_name]

documents = list(collection.find())
df = pd.DataFrame(documents)
client.close()

queries_df = pd.read_csv('notebooks_data/generated_queries.csv')
queries_df = queries_df.iloc[:1000]

train_df, test_df = train_test_split(queries_df, test_size=0.3, random_state=42)
print(f"Training set size: {len(train_df)}")
print(f"Test set size: {len(test_df)}")

from transformers import AutoTokenizer, AutoModel
from peft import LoraConfig, get_peft_model

# Load tokenizer and model without quantization
model_name = "intfloat/e5-large-v2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)  # Standard loading, no quantization

# Define QLoRA configuration
lora_config = LoraConfig(
    r=4,               # Rank of the low-rank matrices
    lora_alpha=16,      # Scaling factor for LoRA updates
    target_modules=["key", "query", "value"],  # Adjust based on model structure if necessary
    lora_dropout=0.1,   # Dropout for LoRA layers
    bias="none"         # Exclude biases to focus on core layers
)

# Wrap the model with QLoRA
model = get_peft_model(model, lora_config)

Training set size: 700
Test set size: 300


In [17]:
import torch

def get_embeddings(text, model, tokenizer):
    inputs = tokenizer(text, return_tensors="pt", padding=True, truncation=True).to("cpu")
    with torch.no_grad():
        embeddings = model(**inputs).last_hidden_state.mean(dim=1).cpu().numpy()
    return embeddings


from sklearn.metrics.pairwise import cosine_similarity

def find_top_news(query, df, model, tokenizer, top_k=10):
    query_embedding = get_embeddings(query, model, tokenizer)

    # Calculate cosine similarity between query and all news embeddings
    similarities = df['e5_embedding_before_tuning'].apply(lambda x: cosine_similarity(query_embedding, x)[0][0])
    
    # Add similarity scores to the DataFrame
    df['e5_similarity'] = similarities
    
    # Sort by similarity and return the top_k results
    top_matches = df.nlargest(top_k, 'e5_similarity')
    
    # Clean up by dropping the 'similarity' column
    df.drop(columns=['e5_similarity'], inplace=True)
    
    return top_matches[['content', 'e5_similarity']]


In [18]:
from torch.utils.data import Dataset, DataLoader

class QueryNewsDataset(Dataset):
    def __init__(self, df, tokenizer, max_length=128):
        self.queries = df['query'].tolist()
        self.news = df['news_content'].tolist()
        self.tokenizer = tokenizer
        self.max_length = max_length
    
    def __len__(self):
        return len(self.queries)
    
    def __getitem__(self, idx):
        query_text = self.queries[idx]
        news_text = self.news[idx]
        
        query_inputs = self.tokenizer(query_text, padding='max_length', truncation=True, max_length=self.max_length, return_tensors="pt")
        news_inputs = self.tokenizer(news_text, padding='max_length', truncation=True, max_length=self.max_length, return_tensors="pt")
        
        return {
            'query_input_ids': query_inputs['input_ids'].squeeze(0),
            'query_attention_mask': query_inputs['attention_mask'].squeeze(0),
            'news_input_ids': news_inputs['input_ids'].squeeze(0),
            'news_attention_mask': news_inputs['attention_mask'].squeeze(0)
        }

# Create DataLoader for fine-tuning
train_loader = DataLoader(QueryNewsDataset(test_df, tokenizer), batch_size=8, shuffle=True)




import torch
from torch.nn import functional as F
from transformers import AdamW

# Optimizer
optimizer = AdamW(model.parameters(), lr=2e-5)

# Custom similarity loss function
def similarity_loss(embedding1, embedding2):
    cos_sim = F.cosine_similarity(embedding1, embedding2)
    return 1 - cos_sim.mean()

# Fine-tuning loop with progress updates
model.train()
for epoch in range(3):  # Adjust the number of epochs as needed
    total_loss = 0
    for i, batch in enumerate(train_loader):
        optimizer.zero_grad()
        
        # Query embeddings
        query_embeddings = model(
            input_ids=batch['query_input_ids'],
            attention_mask=batch['query_attention_mask']
        ).last_hidden_state.mean(dim=1)
        
        # News content embeddings
        news_embeddings = model(
            input_ids=batch['news_input_ids'],
            attention_mask=batch['news_attention_mask']
        ).last_hidden_state.mean(dim=1)
        
        # Calculate similarity loss and optimize
        loss = similarity_loss(query_embeddings, news_embeddings)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        
        # Print progress within each epoch
        if (i + 1) % 10 == 0:
            percent_complete = ((i + 1) / len(train_loader)) * 100
            print(f"Epoch {epoch + 1} - Processed {percent_complete:.0f}% -> {i + 1} out of {len(train_loader)} batches.")

    avg_loss = total_loss / len(train_loader)
    print(f"Epoch {epoch + 1} completed with average loss: {avg_loss:.4f}")

/Users/who/projects/task1/venv/lib/python3.11/site-packages/transformers/optimization.py:591: FutureWarning: This implementation of AdamW is deprecated and will be removed in a future version. Use the PyTorch implementation torch.optim.AdamW instead, or set `no_deprecation_warning=True` to disable this warning
  warnings.warn(


Epoch 1 - Processed 26% -> 10 out of 38 batches.
Epoch 1 - Processed 53% -> 20 out of 38 batches.
Epoch 1 - Processed 79% -> 30 out of 38 batches.
Epoch 1 completed with average loss: 0.1322
Epoch 2 - Processed 26% -> 10 out of 38 batches.
Epoch 2 - Processed 53% -> 20 out of 38 batches.
Epoch 2 - Processed 79% -> 30 out of 38 batches.
Epoch 2 completed with average loss: 0.1128
Epoch 3 - Processed 26% -> 10 out of 38 batches.
Epoch 3 - Processed 53% -> 20 out of 38 batches.
Epoch 3 - Processed 79% -> 30 out of 38 batches.
Epoch 3 completed with average loss: 0.0849


In [19]:
# Post-fine-tuning similarity calculation with progress updates
similarities_after = []
model.eval()
for i, row in test_df.iterrows():
    query_embedding = get_embeddings(row['query'], model, tokenizer)
    content_embedding = get_embeddings(row['news_content'], model, tokenizer)
    similarity = cosine_similarity(query_embedding, content_embedding)[0][0]
    similarities_after.append(similarity)
    
    # Print progress
    if (i + 1) % 100 == 0:
        percent_complete = ((i + 1) / len(test_df)) * 100
        print(f"Processed {percent_complete:.0f}% -> {i + 1} out of {len(test_df)} rows.")

print("Average Post-Fine-Tuning Similarity:", sum(similarities_after) / len(similarities_after))

df['e5_embedding_before_tuning'] = df['content'].apply(lambda x: get_embeddings(x, model, tokenizer))

test_queries = ["Products for Color-blind people","JEEP off-road vehicles","What is the origin of Bamba snack","Recommended diet and exercise habits for managing obesity","Keep your children safe on the internet","What are the lates news about AI ethics?","How the war between Israel and Hamas is affecting global politics?","Latest trends in the fashion industry","Interest rates","biden vs trump","Best Marvel movie this year","Northern Lights"]
for user_query in test_queries:
    print(f"\n\nQuery: {user_query}")
    top_news = find_top_news(user_query, df, model, tokenizer)
    for _, row in top_news.iterrows():
                print(f"\033[96m Similarity: {row['e5_similarity']:.3f}::\033[0m {row['content']}")

Processed 300% -> 900 out of 300 rows.
Processed 167% -> 500 out of 300 rows.
Processed 100% -> 300 out of 300 rows.
Processed 200% -> 600 out of 300 rows.
Processed 67% -> 200 out of 300 rows.
Processed 267% -> 800 out of 300 rows.
Average Post-Fine-Tuning Similarity: 0.9213558


Query: Products for Color-blind people
 Similarity: 0.861:: Ethnic bias in health devices 'puts patients at risk'. The review focuses on devices frequently used in the NHS, like pulse oximeters and skin cancer apps.
 Similarity: 0.853:: 'I’ve invented an alternative to uncomfortable smear tests'. Sânziana Foia's non-invasive test detects harmful virus strains and delivers fast results at home.
 Similarity: 0.852:: Cost of living: How can I make extra income?. Some people have been earning thousands selling clothes and other items since they were in school.
 Similarity: 0.851:: What is behind the TikTok thirst for Stanley water cups?. People are camping outside supermarkets in the US to try to buy the latest i

# 1K queries, Rank 4, alpha 4

In [20]:
import pandas as pd
from sklearn.model_selection import train_test_split

from pymongo import MongoClient
import pandas as pd

# MongoDB connection details
mongo_uri = "mongodb://localhost:27017/"
db_name = "task1"
collection_name = "bbc_news"

# Connect to MongoDB
client = MongoClient(mongo_uri)
db = client[db_name]
collection = db[collection_name]

documents = list(collection.find())
df = pd.DataFrame(documents)
client.close()

queries_df = pd.read_csv('notebooks_data/generated_queries.csv')
queries_df = queries_df.iloc[:1000]

train_df, test_df = train_test_split(queries_df, test_size=0.3, random_state=42)
print(f"Training set size: {len(train_df)}")
print(f"Test set size: {len(test_df)}")

from transformers import AutoTokenizer, AutoModel
from peft import LoraConfig, get_peft_model

# Load tokenizer and model without quantization
model_name = "intfloat/e5-large-v2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)  # Standard loading, no quantization

# Define QLoRA configuration
lora_config = LoraConfig(
    r=4,               # Rank of the low-rank matrices
    lora_alpha=4,      # Scaling factor for LoRA updates
    target_modules=["key", "query", "value"],  # Adjust based on model structure if necessary
    lora_dropout=0.1,   # Dropout for LoRA layers
    bias="none"         # Exclude biases to focus on core layers
)

# Wrap the model with QLoRA
model = get_peft_model(model, lora_config)

Training set size: 700
Test set size: 300


In [21]:
import torch

def get_embeddings(text, model, tokenizer):
    inputs = tokenizer(text, return_tensors="pt", padding=True, truncation=True).to("cpu")
    with torch.no_grad():
        embeddings = model(**inputs).last_hidden_state.mean(dim=1).cpu().numpy()
    return embeddings


from sklearn.metrics.pairwise import cosine_similarity

def find_top_news(query, df, model, tokenizer, top_k=10):
    query_embedding = get_embeddings(query, model, tokenizer)

    # Calculate cosine similarity between query and all news embeddings
    similarities = df['e5_embedding_before_tuning'].apply(lambda x: cosine_similarity(query_embedding, x)[0][0])
    
    # Add similarity scores to the DataFrame
    df['e5_similarity'] = similarities
    
    # Sort by similarity and return the top_k results
    top_matches = df.nlargest(top_k, 'e5_similarity')
    
    # Clean up by dropping the 'similarity' column
    df.drop(columns=['e5_similarity'], inplace=True)
    
    return top_matches[['content', 'e5_similarity']]


In [22]:
from torch.utils.data import Dataset, DataLoader

class QueryNewsDataset(Dataset):
    def __init__(self, df, tokenizer, max_length=128):
        self.queries = df['query'].tolist()
        self.news = df['news_content'].tolist()
        self.tokenizer = tokenizer
        self.max_length = max_length
    
    def __len__(self):
        return len(self.queries)
    
    def __getitem__(self, idx):
        query_text = self.queries[idx]
        news_text = self.news[idx]
        
        query_inputs = self.tokenizer(query_text, padding='max_length', truncation=True, max_length=self.max_length, return_tensors="pt")
        news_inputs = self.tokenizer(news_text, padding='max_length', truncation=True, max_length=self.max_length, return_tensors="pt")
        
        return {
            'query_input_ids': query_inputs['input_ids'].squeeze(0),
            'query_attention_mask': query_inputs['attention_mask'].squeeze(0),
            'news_input_ids': news_inputs['input_ids'].squeeze(0),
            'news_attention_mask': news_inputs['attention_mask'].squeeze(0)
        }

# Create DataLoader for fine-tuning
train_loader = DataLoader(QueryNewsDataset(test_df, tokenizer), batch_size=8, shuffle=True)




import torch
from torch.nn import functional as F
from transformers import AdamW

# Optimizer
optimizer = AdamW(model.parameters(), lr=2e-5)

# Custom similarity loss function
def similarity_loss(embedding1, embedding2):
    cos_sim = F.cosine_similarity(embedding1, embedding2)
    return 1 - cos_sim.mean()

# Fine-tuning loop with progress updates
model.train()
for epoch in range(3):  # Adjust the number of epochs as needed
    total_loss = 0
    for i, batch in enumerate(train_loader):
        optimizer.zero_grad()
        
        # Query embeddings
        query_embeddings = model(
            input_ids=batch['query_input_ids'],
            attention_mask=batch['query_attention_mask']
        ).last_hidden_state.mean(dim=1)
        
        # News content embeddings
        news_embeddings = model(
            input_ids=batch['news_input_ids'],
            attention_mask=batch['news_attention_mask']
        ).last_hidden_state.mean(dim=1)
        
        # Calculate similarity loss and optimize
        loss = similarity_loss(query_embeddings, news_embeddings)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        
        # Print progress within each epoch
        if (i + 1) % 10 == 0:
            percent_complete = ((i + 1) / len(train_loader)) * 100
            print(f"Epoch {epoch + 1} - Processed {percent_complete:.0f}% -> {i + 1} out of {len(train_loader)} batches.")

    avg_loss = total_loss / len(train_loader)
    print(f"Epoch {epoch + 1} completed with average loss: {avg_loss:.4f}")

/Users/who/projects/task1/venv/lib/python3.11/site-packages/transformers/optimization.py:591: FutureWarning: This implementation of AdamW is deprecated and will be removed in a future version. Use the PyTorch implementation torch.optim.AdamW instead, or set `no_deprecation_warning=True` to disable this warning
  warnings.warn(


Epoch 1 - Processed 26% -> 10 out of 38 batches.
Epoch 1 - Processed 53% -> 20 out of 38 batches.
Epoch 1 - Processed 79% -> 30 out of 38 batches.
Epoch 1 completed with average loss: 0.1381
Epoch 2 - Processed 26% -> 10 out of 38 batches.
Epoch 2 - Processed 53% -> 20 out of 38 batches.
Epoch 2 - Processed 79% -> 30 out of 38 batches.
Epoch 2 completed with average loss: 0.1339
Epoch 3 - Processed 26% -> 10 out of 38 batches.
Epoch 3 - Processed 53% -> 20 out of 38 batches.
Epoch 3 - Processed 79% -> 30 out of 38 batches.
Epoch 3 completed with average loss: 0.1261


In [23]:
# Post-fine-tuning similarity calculation with progress updates
similarities_after = []
model.eval()
for i, row in test_df.iterrows():
    query_embedding = get_embeddings(row['query'], model, tokenizer)
    content_embedding = get_embeddings(row['news_content'], model, tokenizer)
    similarity = cosine_similarity(query_embedding, content_embedding)[0][0]
    similarities_after.append(similarity)
    
    # Print progress
    if (i + 1) % 100 == 0:
        percent_complete = ((i + 1) / len(test_df)) * 100
        print(f"Processed {percent_complete:.0f}% -> {i + 1} out of {len(test_df)} rows.")

print("Average Post-Fine-Tuning Similarity:", sum(similarities_after) / len(similarities_after))

df['e5_embedding_before_tuning'] = df['content'].apply(lambda x: get_embeddings(x, model, tokenizer))

test_queries = ["Products for Color-blind people","JEEP off-road vehicles","What is the origin of Bamba snack","Recommended diet and exercise habits for managing obesity","Keep your children safe on the internet","What are the lates news about AI ethics?","How the war between Israel and Hamas is affecting global politics?","Latest trends in the fashion industry","Interest rates","biden vs trump","Best Marvel movie this year","Northern Lights"]
for user_query in test_queries:
    print(f"\n\nQuery: {user_query}")
    top_news = find_top_news(user_query, df, model, tokenizer)
    for _, row in top_news.iterrows():
                print(f"\033[96m Similarity: {row['e5_similarity']:.3f}::\033[0m {row['content']}")

Processed 300% -> 900 out of 300 rows.
Processed 167% -> 500 out of 300 rows.
Processed 100% -> 300 out of 300 rows.
Processed 200% -> 600 out of 300 rows.
Processed 67% -> 200 out of 300 rows.
Processed 267% -> 800 out of 300 rows.
Average Post-Fine-Tuning Similarity: 0.8682825


Query: Products for Color-blind people
 Similarity: 0.799:: Cost of living: How can I make extra income?. Some people have been earning thousands selling clothes and other items since they were in school.
 Similarity: 0.799:: Ethnic bias in health devices 'puts patients at risk'. The review focuses on devices frequently used in the NHS, like pulse oximeters and skin cancer apps.
 Similarity: 0.790:: 'I’ve invented an alternative to uncomfortable smear tests'. Sânziana Foia's non-invasive test detects harmful virus strains and delivers fast results at home.
 Similarity: 0.789:: Electioncast: Laura Kuenssberg and Adam Fleming answer your questions. Adam and Laura answer your election questions
 Similarity: 0.78